In [1]:
import pandas as pd
import yfinance as yf

import io
import requests

In [2]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
tables = pd.read_html(io.StringIO(response.text))

sp500_df = tables[0]

In [3]:
sp500_df.sample(3)

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
448,TRV,Travelers Companies (The),Financials,Property & Casualty Insurance,"New York City, New York",2002-08-21,86312,1853
30,AMP,Ameriprise Financial,Financials,Asset Management & Custody Banks,"Minneapolis, Minnesota",2005-10-03,820027,1894
223,HAL,Halliburton,Energy,Oil & Gas Equipment & Services,"Houston, Texas",1957-03-04,45012,1919


# Question 1: [Index] S&P 500 Stocks Added to the Index

In [4]:
sp500_df['Date added'] = pd.to_datetime(sp500_df['Date added'])
sp500_df['year_added'] = sp500_df['Date added'].dt.year

In [5]:
sp500_df.groupby('year_added').size().sort_values(ascending=False)[:10]

year_added
1957    52
2017    22
2019    21
2016    21
2025    18
2024    16
2008    16
2023    15
2022    15
1997    14
dtype: int64

# Q2 Question 2. [Macro] Indexes YTD (as of 21 August 2026)

In [6]:
start_date = '2026-01-01'
end_date = '2026-08-21'

indices = [
    '^GSPC', '000001.SS', '^HSI', '^AXJO', '^NSEI', '^GSPTSE', '^GDAXI', '^FTSE', '^N225', '^MXX', '^BVSP',
    ]


data = yf.download(
    tickers=indices,
    start=start_date,
    end=end_date,
    interval='1d',
    progress=False
)

In [7]:
data = data['Close']
data.sample(3)

Ticker,000001.SS,^AXJO,^BVSP,^FTSE,^GDAXI,^GSPC,^GSPTSE,^HSI,^MXX,^N225,^NSEI
Date,,,,,,,,,,,
2026-07-20,3796.281006,8791.299805,173371.0,10524.799805,24846.689453,7443.279785,34960.300781,25143.050781,66122.781250,NaN,24238.500000
2026-04-23,4093.250000,8793.400391,191378.0,10457.000000,24155.449219,7108.399902,33912.898438,25915.199219,68631.156250,59140.230469,24173.050781
2026-01-19,4114.000000,8874.500000,164849.0,10195.400391,24959.060547,NaN,33091.000000,26563.900391,67467.820312,53583.570312,25585.500000


In [8]:
data = data.ffill()
data[data.isna().any(axis=1)]

Ticker,000001.SS,^AXJO,^BVSP,^FTSE,^GDAXI,^GSPC,^GSPTSE,^HSI,^MXX,^N225,^NSEI
Date,,,,,,,,,,,
2026-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26146.550781
2026-01-02,NaN,8727.799805,160539.0,9951.099609,24539.339844,6858.470215,31883.400391,26338.470703,64141.359375,NaN,26328.550781


In [9]:
first_prices = data.bfill().iloc[0]
last_prices = data.ffill().iloc[-1]

# YTD
ytd_total = (last_prices / first_prices) - 1

ytd_table = (ytd_total * 100).round(2).sort_values(ascending=False).to_frame(name='YTD Return (%)')
print(ytd_table)

           YTD Return (%)
Ticker                   
^N225               27.75
^GSPTSE             14.06
^GSPC               11.41
^FTSE                8.01
^GDAXI               5.88
^BVSP                4.60
^AXJO                4.08
^MXX                 0.32
^HSI                -2.43
000001.SS           -2.97
^NSEI               -7.32


# Question 3. [Index] S&P 500 Market Corrections Analysis

In [10]:
start_date = '1950-01-01'
end_date = pd.Timestamp.today().strftime('%Y-%m-%d')

indices = ['^GSPC']


data = yf.download(
    tickers=indices,
    start=start_date,
    end=end_date,
    interval='1d',
    progress=False
)

data = data['Close']
data.sample(3)

Ticker,^GSPC
Date,
1979-11-27,106.379997
1960-09-14,55.439999
1981-06-10,132.320007


In [11]:
price = data['^GSPC']
cummax = price.cummax()
is_ath = (price == cummax) & (price > cummax.shift(1))
is_ath.iloc[0] = True

data['cummax'] = cummax
data['is_ath'] = is_ath

data.head(10)

Ticker,^GSPC,cummax,is_ath
Date,,,
1950-01-03,16.660000,16.66,True
1950-01-04,16.850000,16.85,True
1950-01-05,16.930000,16.93,True
1950-01-06,16.980000,16.98,True
1950-01-09,17.080000,17.08,True
1950-01-10,17.030001,17.08,False
1950-01-11,17.090000,17.09,True
1950-01-12,16.760000,17.09,False
1950-01-13,16.670000,17.09,False


In [12]:
print(data.loc['2007-10-09':'2009-03-09']['^GSPC'].min())
print(data.loc['2007-10-09':'2009-03-09']['is_ath'].sum())

676.530029296875
1


In [13]:
ath_dates = data.index[data['is_ath']].tolist()
print(f"1. ATH points: {len(ath_dates)}")

# 2. minimum between ATHs
records = []

for i in range(len(ath_dates) - 1):
    d_start = ath_dates[i]
    d_end = ath_dates[i + 1]
    
    prices_between = data.loc[d_start:d_end, '^GSPC'].iloc[1:-1]
    
    if len(prices_between) == 0:
        continue
    
    ath_price = float(data.loc[d_start, '^GSPC'])
    min_price = float(prices_between.min())
    min_date = prices_between.idxmin()
    
    records.append({
        'prev_ath_date': d_start,
        'next_ath_date': d_end,
        'prev_ath_price': ath_price,
        'min_date': min_date,
        'min_price': min_price
    })

pairs_df = pd.DataFrame(records)

1. ATH points: 1510


In [14]:
pairs_df['drawdown_pct'] = (
    (pairs_df['prev_ath_price'] - pairs_df['min_price']) / pairs_df['prev_ath_price'] * 100.0
)

pairs_df['duration_days'] = (pairs_df['next_ath_date'] - pairs_df['prev_ath_date']).dt.days

In [15]:
print(f"drawdowns between ATH: {len(pairs_df)}")
pairs_df.head(10)

drawdowns between ATH: 687


,prev_ath_date,next_ath_date,prev_ath_price,min_date,min_price,drawdown_pct,duration_days
0,1950-01-09,1950-01-11,17.080000,1950-01-10,17.030001,0.292736,2
1,1950-01-11,1950-02-02,17.090000,1950-01-13,16.670000,2.457578,22
2,1950-02-06,1950-03-15,17.320000,1950-02-16,16.990000,1.905311,37
3,1950-03-16,1950-03-22,17.490000,1950-03-20,17.440001,0.285873,6
4,1950-03-23,1950-04-05,17.559999,1950-03-31,17.290001,1.537577,13
5,1950-04-10,1950-04-12,17.850000,1950-04-11,17.750000,0.560226,2
6,1950-04-13,1950-04-18,17.980000,1950-04-17,17.879999,0.556176,5
7,1950-04-19,1950-05-01,18.049999,1950-04-26,17.760000,1.606643,12
8,1950-05-01,1950-05-03,18.219999,1950-05-02,18.110001,0.603725,2
9,1950-05-03,1950-05-10,18.270000,1950-05-04,18.120001,0.821016,7


In [16]:
pairs_df[pairs_df['prev_ath_date'] == '2007-10-09']

,prev_ath_date,next_ath_date,prev_ath_price,min_date,min_price,drawdown_pct,duration_days
454,2007-10-09,2013-03-28,1565.150024,2009-03-09,676.530029,56.775388,1997


In [17]:
corrections_5pct = pairs_df[pairs_df['drawdown_pct'] >= 5.0].copy()
print(f">= 5%: {len(corrections_5pct)}")

percentiles = [0.25, 0.50, 0.75]

stats_table = pd.DataFrame({
    'Drawdown (%)': corrections_5pct['drawdown_pct'].quantile(percentiles).values,
    'Duration (Days)': corrections_5pct['duration_days'].quantile(percentiles).values
}, index=['25th Percentile', '50th Percentile (Median)', '75th Percentile'])

median_dd = corrections_5pct['drawdown_pct'].median()
median_dur = corrections_5pct['duration_days'].median()

print(f"\nmedian drawdown of significant market corrections: {median_dd:.2f}%")

>= 5%: 74

median drawdown of significant market corrections: 7.99%


# Question 4. [Stocks] Earnings Surprise Analysis for Amazon (AMZN)

In [18]:
ticker = 'AMZN'
ticker_obj = yf.Ticker(ticker)

result = ticker_obj.get_earnings_dates()
result.index = result.index.tz_localize(None).normalize()
result = result.sort_index()

result.head(3)

,EPS Estimate,Reported EPS,Surprise(%)
Earnings Date,,,
2020-10-29,0.38,0.62,64.25
2021-02-02,0.35,0.70,100.10
2021-04-29,0.47,0.79,67.30


In [19]:
amzn_data = yf.download(
    tickers='AMZN',
    period='max',
    interval='1d',
    progress=False
)

amzn = amzn_data['Close'].squeeze()

amzn.head()

Date
1997-05-15    0.097917
1997-05-16    0.086458
1997-05-19    0.085417
1997-05-20    0.081771
1997-05-21    0.071354
Name: AMZN, dtype: float64

In [20]:
amzn = amzn.sort_index()

close_day1 = amzn.shift(1)
close_day3 = amzn.shift(-1)

return_2d = (close_day3 / close_day1) - 1

df_amzn = pd.DataFrame({
    'Close_Day1': close_day1,
    'Close_Day2': amzn,
    'Close_Day3': close_day3,
    'return_2d': return_2d,
    'return_2d_pct': return_2d * 100
})

print(df_amzn.iloc[-5:-1])

            Close_Day1  Close_Day2  Close_Day3  return_2d  return_2d_pct
Date                                                                    
2026-09-04  258.899994  258.510010  256.970001  -0.007455      -0.745459
2026-09-08  258.510010  256.970001  252.399994  -0.023636      -2.363551
2026-09-09  256.970001  252.399994  251.889999  -0.019769      -1.976885
2026-09-10  252.399994  251.889999  256.779999   0.017353       1.735343


 # Merge estimation and real prices

In [21]:
merged = result.join(df_amzn[['return_2d', 'return_2d_pct']], how='inner')
merged = merged.dropna(subset=['Reported EPS', 'Surprise(%)'])
merged.head()

,EPS Estimate,Reported EPS,Surprise(%),return_2d,return_2d_pct
Earnings Date,,,,,
2020-10-29,0.38,0.62,64.25,-0.040038,-4.003764
2021-02-02,0.35,0.70,100.10,-0.009079,-0.907901
2021-04-29,0.47,0.79,67.30,0.002579,0.257915
2021-07-29,0.61,0.76,23.06,-0.083389,-8.338937
2021-10-28,0.44,0.31,-31.21,-0.005913,-0.591301


In [22]:
positive_surprises = merged[merged['Surprise(%)'] > 0].copy()
median_return_pct = positive_surprises['return_2d_pct'].median()

correlation_matrix = positive_surprises[['Surprise(%)', 'return_2d_pct']].corr()
print("\nCorrelation Matrix:")
print(correlation_matrix.round(4))


Correlation Matrix:
               Surprise(%)  return_2d_pct
Surprise(%)         1.0000         0.3306
return_2d_pct       0.3306         1.0000


# Question 5. [Exploratory, optional] Brainstorm potential idea for your capstone project

I'm going to build ranking model for the global gambling/betting sector

# Question 6. [Exploratory, optional] Investigate new metrics

- BETZ (Roundhill Sports Betting & iGaming ETF)
- Google Trends
- EDGAR (Electronic Data Gathering, Analysis, and Retrieval)
- price-based technicals